# Wikimedia Event Streaming Pipeline

This notebook defines the `produce(max_events)` function that connects to the [Wikimedia RecentChanges SSE endpoint](https://stream.wikimedia.org/v2/stream/recentchange), collects live edit events, and returns them as a pandas DataFrame. It is designed to be called by the **wikimedia_bronze** notebook (via `%run`), which converts the DataFrame to Spark, persists rows to the bronze Delta table (`workspace.bronze.wikimedia_bronze`), and hands them off to the Silver layer.

## Pipeline Role

1. **Produce** — Connect to the Wikimedia SSE stream, collect up to `max_events` recent-change events, and return them as a pandas DataFrame.
2. **Downstream** — The `wikimedia_bronze` notebook calls `produce()`, converts the result to a Spark DataFrame, and writes it to the bronze Delta table for downstream Silver-layer CDC merge.

## Key Parameters

| Parameter     | Description                                              |
|---------------|---------------------------------------------------------|
| `STREAM_URL`  | Wikimedia SSE endpoint for recent-change events          |
| `max_events`  | Maximum number of events to collect per invocation      |


### convert_dtypes()
convert_dtypes() is a pandas DataFrame/Series method that looks at each column’s actual values and converts it to the “best” dtype that supports pandas’ nullable missing-value marker (pd.NA), instead of whatever generic dtype pandas guessed when the DataFrame was first built.

### Why it matters here: 
plain pandas numeric dtypes (int64, float64) are leftovers from NumPy and can’t represent missing values in an integer column — NaN only exists for floats. \
So if a column has even one missing value, pandas silently upcasts the whole column to float64, even if every actual value is a whole number. That’s exactly what happened to id/log_id: a few events missing log_id forced the entire column to float64, so real integers like 2636605573 got stored as 2.636605573e9.

In [0]:
import json
import logging
import requests
import pandas as pd
from datetime import datetime

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")

STREAM_URL = "https://stream.wikimedia.org/v2/stream/recentchange"
HEADERS    = {"User-Agent": "rana-databricks-demo/1.0 (rana@ghazzi.com)"}

output_dir="/Volumes/workspace/bronze/landing_zone/wikimedia/"
def produce(max_events):
    # Read Wikimedia SSE stream, collect events, return as a DataFrame
    resp = requests.get(STREAM_URL, stream=True, timeout=30, headers=HEADERS)
    print("status:", resp.status_code)
    logging.info(f"stream connected, status={resp.status_code}")

    events = []
    for raw_line in resp.iter_lines():
        if not raw_line:
            continue
        decoded = raw_line.decode("utf-8")
        if not decoded.startswith("data: "):
            continue
        payload = decoded[len("data: "):]
        event = json.loads(payload)
        events.append(event)
        if len(events) >= max_events:
            break

    resp.close()
    spark_df = pd.DataFrame(events).convert_dtypes()
    print(f"[produce] wrote {len(spark_df)} events")
    logging.info(f"produce complete: {len(spark_df)} events collected")
    return spark_df

def save_json(df, output_dir):
    # Write the DataFrame to a timestamped JSON-lines file in the landing zone
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = f"{output_dir}/wikimedia_events_{timestamp}.json"
    df.to_json(path, orient="records", lines=True)
    print(f"[save_json] wrote {len(df)} rows to {path}")
    logging.info(f"save_json complete: {path}")
    return path


spark_df = produce(20)
print(spark_df.dtypes)
print(spark_df[["id", "log_id"]].to_json(orient="records", lines=True))
landing_path = save_json(df=spark_df, output_dir=output_dir)
